# DATA209 — Advanced Exploratory Data Analysis
# Practical P11-12 · Grouping — K-means, silhouette and pair plots

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 6 · Module 2 · CO2

---

**Objective.** Use clustering exploratorily to discover structure, evaluate whether that structure is real, and profile and visualise the groups found.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 — the dataset and the target.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P11-12 — Grouping — K-means, silhouette and pair plots

### Implement K-means for structure discovery

K-means minimises Euclidean distance, so **features must be scaled first** or the column with the
largest range decides the clusters by itself. In this course clustering is exploratory: a way of
seeing structure, not a predictive model.

In [ ]:
# ---- Prepare and scale --------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_cols = ["Administrative", "Administrative_Duration", "Informational",
                "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
                "BounceRates", "ExitRates", "PageValues"]

Xc = df[cluster_cols].copy()
print("Raw ranges — why scaling is not optional:")
print(Xc.agg(["min", "max", "std"]).T.round(2).to_string())

scaler = StandardScaler()
Xs = scaler.fit_transform(Xc)
print("\nAfter standardising: mean ~0, std ~1 for every column.")

In [ ]:
# ---- Choose k: inertia and silhouette ----------------------------------
ks, inertias, sils = range(2, 9), [], []

for k in ks:
    km  = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    lab = km.fit_predict(Xs)
    inertias.append(km.inertia_)
    # silhouette on a sample keeps this fast on 12k rows
    sils.append(silhouette_score(Xs, lab, sample_size=4000, random_state=RANDOM_STATE))

choice = pd.DataFrame({"k": list(ks), "inertia": inertias, "silhouette": sils})
print(choice.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(list(ks), inertias, marker="o", color="#1F6F6B")
axes[0].set_title("Elbow — within-cluster sum of squares"); axes[0].set_xlabel("k")
axes[1].plot(list(ks), sils, marker="o", color="#B5432E")
axes[1].set_title("Mean silhouette"); axes[1].set_xlabel("k")
plt.tight_layout(); plt.show()

best_k = int(choice.loc[choice["silhouette"].idxmax(), "k"])
print(f"Highest silhouette at k = {best_k} ({max(sils):.3f})")

### Silhouette score (basic)

Silhouette compares how close a point is to its own cluster versus the nearest other cluster.
It runs from -1 to +1:

| Score | Reading |
|---|---|
| > 0.7 | strong, well-separated clusters |
| 0.5 – 0.7 | reasonable structure |
| 0.25 – 0.5 | weak; the clusters overlap |
| < 0.25 | no substantial structure — say so |

**Reporting a weak score honestly is the correct answer.** Manufacturing clusters that are not
there is not.

In [ ]:
# ---- Fit the chosen solution and inspect the silhouette ----------------
from sklearn.metrics import silhouette_samples

km     = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
labels = km.fit_predict(Xs)
df_cl  = df.copy()
df_cl["cluster"] = labels

samp_idx = np.random.RandomState(RANDOM_STATE).choice(len(Xs), 4000, replace=False)
sv = silhouette_samples(Xs[samp_idx], labels[samp_idx])
overall = sv.mean()

print(f"Overall mean silhouette: {overall:.3f}")
per_cluster = pd.DataFrame({"cluster": labels[samp_idx], "silhouette": sv}) \
                .groupby("cluster")["silhouette"].agg(["mean", "size"])
print(per_cluster.round(3).to_string())

verdict = ("strong" if overall > .7 else "reasonable" if overall > .5
           else "weak — clusters overlap" if overall > .25 else "no substantial structure")
print(f"\nVerdict: {verdict}.")
print("Report this honestly in your write-up rather than presenting the clusters as clean segments.")

In [ ]:
# ---- Profile the clusters ----------------------------------------------
profile = df_cl.groupby("cluster")[cluster_cols].median()
profile["sessions"]      = df_cl["cluster"].value_counts().sort_index()
profile["share_%"]       = (profile["sessions"] / len(df_cl) * 100).round(1)
profile["conversion_%"]  = (df_cl.groupby("cluster")[TARGET].mean() * 100).round(2)
print("Cluster profile (medians)")
print(profile.T.round(2).to_string())

overall_rate = df_cl[TARGET].mean() * 100
print(f"\nOverall conversion rate: {overall_rate:.2f}%")
print("\nName each cluster from its profile. A cluster you cannot describe in one sentence")
print("should not appear in your report.")

In [ ]:
# ---- Stability check: does the structure survive a different seed? -----
stability = []
for seed in [0, 7, 42, 123]:
    lab = KMeans(n_clusters=best_k, n_init=10, random_state=seed).fit_predict(Xs)
    stability.append({
        "seed": seed,
        "silhouette": silhouette_score(Xs, lab, sample_size=4000, random_state=0),
        "largest_cluster_%": round(pd.Series(lab).value_counts(normalize=True).max()*100, 1),
    })
print(pd.DataFrame(stability).round(3).to_string(index=False))
print("\nStructure that changes materially with the seed was never there.")

### Visualize groups

Nine dimensions cannot be plotted directly, so project onto the first two principal components.
PCA is covered properly in P27-28; here it is used only as a viewing device.

In [ ]:
# ---- 2-D projection coloured by cluster and by outcome -----------------
from sklearn.decomposition import PCA

proj = PCA(n_components=2, random_state=RANDOM_STATE).fit(Xs)
P2   = proj.transform(Xs)
samp = np.random.RandomState(RANDOM_STATE).choice(len(P2), 4000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.scatterplot(x=P2[samp, 0], y=P2[samp, 1], hue=labels[samp],
                palette="deep", s=10, alpha=0.6, ax=axes[0], legend="full")
axes[0].set_title(f"K-means clusters (k={best_k}) on PC1/PC2")

sns.scatterplot(x=P2[samp, 0], y=P2[samp, 1], hue=df[TARGET].values[samp],
                palette=["#C9D4DB", "#B5432E"], s=10, alpha=0.6, ax=axes[1])
axes[1].set_title("Same projection, coloured by actual outcome")

for a in axes:
    a.set_xlabel(f"PC1 ({proj.explained_variance_ratio_[0]*100:.1f}% var)")
    a.set_ylabel(f"PC2 ({proj.explained_variance_ratio_[1]*100:.1f}% var)")
plt.tight_layout(); plt.show()

print(f"PC1 + PC2 capture {proj.explained_variance_ratio_[:2].sum()*100:.1f}% of the variance.")
print("Compare the two panels: do the discovered clusters align with the outcome, or not?")

### Pair plots

A pair plot is a matrix of scatterplots for every variable pair, with distributions on the
diagonal. It is the densest single view of multivariate structure — and slow, so **always sample**.

In [ ]:
# ---- Pair plot ----------------------------------------------------------
pair_cols = ["ProductRelated", "ProductRelated_Duration", "BounceRates",
             "ExitRates", "PageValues"]

pair_df = df_cl.sample(1500, random_state=RANDOM_STATE)[pair_cols + ["cluster"]]
pair_df["cluster"] = pair_df["cluster"].astype(str)

g = sns.pairplot(pair_df, hue="cluster", corner=True, diag_kind="kde",
                 plot_kws=dict(s=8, alpha=0.35), height=1.5)
g.figure.suptitle("Pair plot by cluster (1,500-row sample)", y=1.01)
plt.show()

print("Read the off-diagonal panels for separation between colours and the diagonal for shape.")
print("Heavy skew makes most panels crowd into one corner — a strong argument for P23-24.")

### Deliverable — P11-12

A notebook containing the k selection evidence, a **named profile for every cluster**, the
stability check across seeds, the 2-D projection, the pair plot, and an **honest verdict on
whether the structure is real**.